In [ ]:
# -*- coding: utf-8 -*-
"""
Random Forest puro para compensação de temperatura
Comparação justa contra Park (sem usar temp_c como feature)

Treino:  0, 20, 40, 60 °C
Teste:  -10, 10, 30, 50, 70 °C
Referência: 20 °C

Melhorias:
- RF mais potente (mais árvores, profundidade ilimitada, max_features=None)
- Chunk menor (20 → mais granularidade na curva)
- Features extras internas da curva (média, std, quantis) — não dependem de temp_c
"""

import re, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore", category=UserWarning)

# =========================
# PARÂMETROS
# =========================
REF_TEMP      = 20
FREQ_MIN_KHZ  = 30
FREQ_MAX_KHZ  = 50
PKL_TREINO    = "base_treino.pkl"
PKL_PROVA     = "base_prova (1).pkl"
CHUNK_SIZE    = 80   # mais granular

TEMPS_TREINO = {0, 20, 40, 60}
TEMPS_PROVA  = {-10, 10, 30, 50, 70}

RF_PARAMS = dict(
    n_estimators=2000,    # mais árvores
    max_depth=None,       # profundidade ilimitada
    min_samples_leaf=1,
    min_samples_split=2,
    max_features=None,    # usa todas as features
    bootstrap=True,
    n_jobs=-1,
    random_state=42,
)

# =========================
# FUNÇÕES AUXILIARES
# =========================
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None:
            fk = f/1e3
            if fmin_khz <= fk <= fmax_khz:
                cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs, float)[order]

def metrics_block(name, y_true, y_pred):
    y1, y2 = y_true.reshape(-1), y_pred.reshape(-1)
    R2   = r2_score(y1, y2)
    RMSE = float(np.sqrt(mean_squared_error(y1, y2)))
    MAE  = float(mean_absolute_error(y1, y2))
    print(f"\n== {name} ==")
    print(f"R²={R2:.4f} | RMSE={RMSE:.4f} | MAE={MAE:.4f}")
    return {"R2":R2,"RMSE":RMSE,"MAE":MAE}

def corr_per_sample(Y, Yhat):
    out=[]
    for i in range(Y.shape[0]):
        y,yh=Y[i],Yhat[i]
        num=((y-y.mean())*(yh-yh.mean())).sum()
        den=np.sqrt(((y-y.mean())**2).sum()*((yh-yh.mean())**2).sum())+1e-12
        out.append(np.clip(num/den,-1,1))
    return np.array(out)

def sam_per_sample(Y,Yhat):
    out=[]
    for i in range(Y.shape[0]):
        y,yh=Y[i],Yhat[i]
        num=(y*yh).sum()
        den=(np.linalg.norm(y)*np.linalg.norm(yh))+1e-12
        cosang=np.clip(num/den,-1,1)
        out.append(np.degrees(np.arccos(cosang)))
    return np.array(out)

def nrmse_per_sample(Y,Yhat):
    out=[]
    for i in range(Y.shape[0]):
        y,yh=Y[i],Yhat[i]
        rmse=np.sqrt(np.mean((y-yh)**2))
        rng=np.max(y)-np.min(y)
        out.append(rmse/(rng+1e-12))
    return np.array(out)
    

# =========================
# FEATURES EXTRAS (sem temp_c)
# =========================
def add_extra_features(Xmat):
    mu  = Xmat.mean(axis=1, keepdims=True)
    sd  = Xmat.std(axis=1,  keepdims=True)
    q95 = np.quantile(Xmat, 0.95, axis=1, keepdims=True)
    q05 = np.quantile(Xmat, 0.05, axis=1, keepdims=True)
    return np.hstack([Xmat, mu, sd, q95, q05])

# =========================
# LOAD BASES
# =========================
base_tr = pd.read_pickle(PKL_TREINO)
base_te = pd.read_pickle(PKL_PROVA)

base_tr = base_tr[base_tr["temp_c"].isin(TEMPS_TREINO)].copy()
base_te = base_te[base_te["temp_c"].isin(TEMPS_PROVA)].copy()

freq_cols_tr, fhz_tr = get_freq_columns(base_tr,FREQ_MIN_KHZ,FREQ_MAX_KHZ)
freq_cols_te, fhz_te = get_freq_columns(base_te,FREQ_MIN_KHZ,FREQ_MAX_KHZ)
common_cols = [c for c in freq_cols_tr if c in freq_cols_te]
fhz = np.array([extract_freq_hz(c) for c in common_cols], float)
order=np.argsort(fhz)
common_cols=[common_cols[i] for i in order]; fhz=fhz[order]

X_tr = base_tr[common_cols].to_numpy(float)
X_te = base_te[common_cols].to_numpy(float)

ref_rows = (base_tr["temp_c"].to_numpy() == REF_TEMP)
assert ref_rows.any(), f"Não há {REF_TEMP} °C no treino!"
y_ref = np.median(base_tr.loc[ref_rows, common_cols].to_numpy(float), axis=0)

Y_tr = (y_ref[None,:] - X_tr)

# Extra features (só da curva)
X_tr_aug = add_extra_features(X_tr)
X_te_aug = add_extra_features(X_te)

X_tr_in,X_val_in,Y_tr_in,Y_val_in=train_test_split(
    X_tr_aug,Y_tr,test_size=0.2,random_state=42,shuffle=True
)

# =========================
# RF EM CHUNKS
# =========================
def fit_predict_rf_chunks(X_tr,Y_tr,X_val,X_te,chunk_size=CHUNK_SIZE):
    n_out=Y_tr.shape[1]
    dtr_pred=np.zeros_like(Y_tr)
    dval_pred=np.zeros((X_val.shape[0],n_out))
    dte_pred =np.zeros((X_te.shape[0],n_out))
    n_chunks=int(np.ceil(n_out/chunk_size))
    print(f"[INFO] Saídas={n_out} | CHUNK={chunk_size} | n_chunks={n_chunks}")

    for start in range(0,n_out,chunk_size):
        end=min(start+chunk_size,n_out)
        cols=slice(start,end)
        rf=RandomForestRegressor(**RF_PARAMS)
        rf.fit(X_tr, Y_tr[:,cols])
        dtr=rf.predict(X_tr)
        dvl=rf.predict(X_val)
        dte=rf.predict(X_te)
        if dtr.ndim==1: dtr=dtr.reshape(-1,1)
        if dvl.ndim==1: dvl=dvl.reshape(-1,1)
        if dte.ndim==1: dte=dte.reshape(-1,1)
        dtr_pred[:,cols]=dtr
        dval_pred[:,cols]=dvl
        dte_pred[:,cols]=dte
        print(f"  Chunk {start}:{end} ok")
    return dtr_pred,dval_pred,dte_pred

t0=time.time()
dtr_pred,dval_pred,dte_pred=fit_predict_rf_chunks(X_tr_in,Y_tr_in,X_val_in,X_te_aug,CHUNK_SIZE)
print(f"[INFO] Tempo RF: {time.time()-t0:.1f}s")

# Reconstrução
Y_tr_hat = X_tr + dtr_pred
Y_val_hat = X_val + dval_pred
Y_te_hat  = X_te + dte_pred

Y_ref_tr  = np.tile(y_ref, (Y_tr_hat.shape[0], 1))
Y_ref_val = np.tile(y_ref, (Y_val_hat.shape[0], 1))
Y_ref_te  = np.tile(y_ref, (Y_te_hat.shape[0], 1))

# =========================
# MÉTRICAS
# =========================
print("\n== MÉTRICAS RF ==")
m_train=metrics_block("TREINO interno",Y_ref_tr,Y_tr_hat)
m_val  =metrics_block("VALIDAÇÃO",Y_ref_val,Y_val_hat)
m_test =metrics_block("PROVA",Y_ref_te,Y_te_hat)

corr_vec=corr_per_sample(Y_ref_te,Y_te_hat)
sam_vec =sam_per_sample(Y_ref_te,Y_te_hat)
nrmse_v =nrmse_per_sample(Y_ref_te,Y_te_hat)
print(f"Corr={corr_vec.mean():.4f} | SAM°={sam_vec.mean():.2f} | NRMSE={nrmse_v.mean():.4f}")

# Plot exemplo
if X_te.shape[0]>0:
    plt.figure()
    plt.plot(fhz/1e3, X_te[0],    label="Original")
    plt.plot(fhz/1e3, Y_te_hat[0],label="RF reconstruído")
    plt.plot(fhz/1e3, y_ref,      label=f"Referência {REF_TEMP}°C")
    plt.xlabel("Frequência (kHz)"); plt.ylabel("Re{Z}")
    plt.legend(); plt.grid(); plt.title("Exemplo PROVA idx=0 — RF"); plt.show()


In [ ]:
# =========================
# RECONSTRUÇÃO
# =========================
# Extrair apenas colunas de frequência correspondentes
n_freqs = dtr_pred.shape[1]  # deve ser 20001

# Treino
X_tr_freq  = X_tr_in[:, :n_freqs]
Y_tr_hat   = X_tr_freq + dtr_pred

# Validação
X_val_freq = X_val_in[:, :n_freqs]
Y_val_hat  = X_val_freq + dval_pred

# Teste
X_te_freq  = X_te_aug[:, :n_freqs]
Y_te_hat   = X_te_freq + dte_pred

# Baselines de referência
Y_ref_tr  = np.tile(y_ref, (Y_tr_hat.shape[0], 1))
Y_ref_val = np.tile(y_ref, (Y_val_hat.shape[0], 1))
Y_ref_te  = np.tile(y_ref, (Y_te_hat.shape[0], 1))

# =========================
# MÉTRICAS
# =========================
print("\n== MÉTRICAS RF ==")
m_train = metrics_block("TREINO interno", Y_ref_tr, Y_tr_hat)
m_val   = metrics_block("VALIDAÇÃO",      Y_ref_val, Y_val_hat)
m_test  = metrics_block("PROVA",          Y_ref_te,  Y_te_hat)

corr_vec = corr_per_sample(Y_ref_te, Y_te_hat)
sam_vec  = sam_per_sample(Y_ref_te, Y_te_hat)
nrmse_v  = nrmse_per_sample(Y_ref_te, Y_te_hat)

print(f"Corr={corr_vec.mean():.4f} | SAM°={sam_vec.mean():.2f} | NRMSE={nrmse_v.mean():.4f}")

# =========================
# PLOT EXEMPLO
# =========================
if X_te.shape[0] > 0:
    plt.figure()
    plt.plot(fhz/1e3, X_te[0],     label="Original")
    plt.plot(fhz/1e3, Y_te_hat[0], label="RF reconstruído")
    plt.plot(fhz/1e3, y_ref,       label=f"Referência {REF_TEMP}°C")
    plt.xlabel("Frequência (kHz)")
    plt.ylabel("Re{Z}")
    plt.legend()
    plt.grid()
    plt.title("Exemplo PROVA idx=0 — RF")
    plt.show()